# HyperData ↔ Lumen Integration Tutorial

This notebook demonstrates the full workflow:
1. **Create and push** a HyperData dataset with scientific images
2. **Load** it via Lumen's dataset adapters for training
3. **Train** a model (self-supervised or segmentation)
4. **Push** trained weights back to HyperData with versioning
5. **Pull** weights into a fresh model

## 0. Setup

```bash
uv pip install -e ".[hyperdata]"
```

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from hyperdata import HyperData

from lumen.data.hyperdata import (
    HyperDataImageDataset,
    HyperDataSegmentationDataset,
    WeightManager,
    HYPERDATA_AVAILABLE,
)

print(f"HyperData available: {HYPERDATA_AVAILABLE}")
print(f"PyTorch version: {torch.__version__}")

## 1. Create and Push Dataset to HyperData

Create a local HyperData dataset with synthetic microscopy images,
then push it to a remote HyperData store so that other collaborators
(or a training server) can pull it.

In [ ]:
DATA_DIR = "/tmp/lumen_hyperdata_demo"

# Create a versioned dataset with IceChunk
ds = HyperData(DATA_DIR)

# Synthetic microscopy-like data
np.random.seed(42)
images = np.random.rand(100, 128, 128).astype(np.float32)
masks  = np.random.randint(0, 4, size=(100, 128, 128)).astype(np.int64)
labels = np.random.randint(0, 3, size=(100,)).astype(np.int64)

with ds.transaction("initial data import"):
    ds["images"] = images
    ds["masks"]  = masks
    ds["labels"] = labels

print(f"Dataset keys: {ds.keys()}")
print(f"Images shape: {ds['images'].shape}")
print(f"Masks  shape: {ds['masks'].shape}")

# ---- Push to remote HyperData ----
# Option A: push to S3-compatible storage (RustFS / MinIO / AWS S3)
#   ds.add_remote("origin", "s3://hyperdata-data/datasets/my-experiment")
#   ds.push("origin")
#
# Option B: push via virtual path (requires HyperData API running)
#   ds.push_to_hub("@alice/microscopy-demo")
#
# Both methods require S3 credentials in the environment:
#   S3_ENDPOINT, S3_PORT, S3_ACCESS_KEY, S3_SECRET_KEY, S3_BUCKET
#
# For remote HyperData API, also set:
#   HYPERDATA_ENDPOINT=http://<remote-ip>:8021
print("\nTo push this dataset to remote, run:")
print('  ds.add_remote("origin", "s3://<bucket>/datasets/my-experiment")')
print('  ds.push("origin")')

## 2. Load Data with Lumen Adapters

### 2a. Self-supervised: image-only dataset

In [ ]:
# Load directly from the HyperData object
img_dataset = HyperDataImageDataset(
    ds,
    array_name="images",
    channels=1,       # keep grayscale
    image_size=64,    # resize to 64×64
)

print(f"Dataset length: {len(img_dataset)}")
sample = img_dataset[0]
print(f"Sample keys:    {list(sample.keys())}")
print(f"Image shape:    {sample['image'].shape}")
print(f"Image dtype:    {sample['image'].dtype}")
print(f"Value range:    [{sample['image'].min():.3f}, {sample['image'].max():.3f}]")

### 2b. Segmentation: paired images + masks

In [ ]:
seg_dataset = HyperDataSegmentationDataset(
    ds,
    image_array="images",
    mask_array="masks",
    image_size=64,
    channels=1,
)

sample = seg_dataset[0]
print(f"Image shape: {sample['image'].shape}")  # (1, 64, 64)
print(f"Mask  shape: {sample['mask'].shape}")    # (64, 64)
print(f"Mask  dtype: {sample['mask'].dtype}")     # torch.int64
print(f"Unique mask values: {sample['mask'].unique().tolist()}")

### 2c. Use with PyTorch DataLoader

In [ ]:
loader = torch.utils.data.DataLoader(img_dataset, batch_size=16, shuffle=True)
batch = next(iter(loader))
print(f"Batch image shape: {batch['image'].shape}")  # (16, 1, 64, 64)

## 3. Train a Simple Model

We'll train a tiny CNN to show the end-to-end flow. In practice you'd
use Lumen's `MAETrainer`, `SegmentationTrainer`, etc.

In [ ]:
class TinyCNN(nn.Module):
    """Minimal CNN for demonstration."""
    def __init__(self, in_ch: int = 1, num_classes: int = 4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_ch, 8, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.head = nn.Linear(8, num_classes)

    def forward(self, x):
        f = self.features(x).flatten(1)
        return self.head(f)


model = TinyCNN(in_ch=1, num_classes=4)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# Quick training loop (3 epochs)
loader = torch.utils.data.DataLoader(img_dataset, batch_size=16, shuffle=True)

for epoch in range(3):
    total_loss = 0.0
    for batch in loader:
        imgs = batch["image"]
        # Dummy target for demo (real workflow uses masks/labels)
        targets = torch.randint(0, 4, (imgs.shape[0],))
        optimizer.zero_grad()
        loss = criterion(model(imgs), targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: loss = {total_loss / len(loader):.4f}")

## 4. Push Weights to HyperData

In [ ]:
WEIGHTS_DIR = "/tmp/lumen_hyperdata_demo_weights"

wm = WeightManager(WEIGHTS_DIR)

# Push just the model weights
wm.push_weights(
    model,
    message="TinyCNN after 3 epochs",
    tag="v1.0",
    metrics={"final_loss": total_loss / len(loader)},
)

# Push a full checkpoint (model + optimizer + epoch)
wm.push_checkpoint(
    model, optimizer, epoch=3,
    message="Full checkpoint at epoch 3",
)

print(f"Tags: {wm.list_tags()}")

## 5. Pull Weights into a Fresh Model

In [ ]:
# Create a fresh model with random weights
fresh_model = TinyCNN(in_ch=1, num_classes=4)

# Pull the tagged weights
wm2 = WeightManager(WEIGHTS_DIR)
meta = wm2.pull_weights(fresh_model, tag="v1.0")
print(f"Metadata: {meta}")

# Verify weights match the trained model
for (n1, p1), (n2, p2) in zip(model.named_parameters(), fresh_model.named_parameters()):
    assert torch.allclose(p1, p2), f"Mismatch in {n1}"
print("\nAll weights match — round-trip successful!")

## 6. Resume Training from Checkpoint

In [ ]:
# Restore full checkpoint
resume_model = TinyCNN(in_ch=1, num_classes=4)
resume_opt   = torch.optim.Adam(resume_model.parameters(), lr=1e-3)

ckpt = wm2.pull_checkpoint(resume_model, resume_opt)
print(f"Resumed from epoch {ckpt['epoch']}")
print(f"Checkpoint keys: {list(ckpt.keys())}")

## 7. Loading from a Path (No Existing HyperData Object)

You can also pass a file-system path directly — the adapter will open the
dataset automatically.

In [ ]:
# Pass a path string instead of a HyperData object
dataset_from_path = HyperDataImageDataset(DATA_DIR, array_name="images")
print(f"Loaded {len(dataset_from_path)} images from path")
print(f"Sample shape: {dataset_from_path[0]['image'].shape}")

## Cleanup

In [ ]:
import shutil
shutil.rmtree(DATA_DIR, ignore_errors=True)
shutil.rmtree(WEIGHTS_DIR, ignore_errors=True)
print("Cleaned up temp directories.")